In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import spynal
from spynal.matIO import loadmat
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import os 
from os import listdir
import pickle

## 1. Functions

In [ ]:
## create pseudo-population for decoding analysis

def count_number(labels):
    '''
    Inputs
    ----------
    labels: shape=(4,trial_num) 
    - rows represent t1, t2, r1, r2 respectively
    
    Returns
    ----------
    all_label_number: shape=(4,4) 
    - rows represent t1, t2, r1, r2
    - columns represent trial number for each class of stimuli 0, 1, 2, 3
    
    '''
    all_label_num = []
    for label in labels:
        each_label_num = []
        # count the number of trials that each class of stimuli or rewards has
        for i in range(4):
            each_label_num.append(np.sum(label == i))
        all_label_num.append(np.array(each_label_num).reshape(1,-1))
    return np.concatenate(all_label_num, axis=0)


def create_pseudo_popu(labels_cat, units_cat, min_repeated_trials):
    """
    Create balanced neural pseudopopulations across sessions for decoding
    Target 1 (`t1`) and Target 2 (`t2`) locations.

    Inputs
    ----------
    labels_cat : list
        One label array per session.
        labels_cat[i][0] contains Target 1 labels and labels_cat[i][1]
        contains Target 2 labels. Labels must be 0, 1, 2, or 3.

    units_cat : list
        Neural activity arrays for each session. The first dimension must
        correspond to trials.

    min_repeated_trials : int
        Number of trials randomly sampled from each location class.

    Returns
    -------
    dec_feature : dict
        Pseudopopulation features for `t1` and `t2`, concatenated across
        sessions along the unit dimension.

    dec_label : dict
        Balanced class labels for locations 0, 1, 2, and 3.
    """
    dec_feature = {}
    dec_label = {}
    info = ['t1', 't2']
    for type in info:
        dec_feature[type] = []
        dec_label[type] = np.array([0] * min_repeated_trials + [1] * min_repeated_trials + [2] * min_repeated_trials + [
            3] * min_repeated_trials)

    for i, labels in enumerate(labels_cat):
        for m, type in enumerate(info):
            sel_units = []
            sel_labels = []
            for n in range(4):
                class_idx = np.where(labels[m] == n)[0]  # trial idx where class number = n
                sel_class_idx = np.random.choice(class_idx, min_repeated_trials, replace=False)
                sel_units.append(units_cat[i][sel_class_idx])
                sel_labels.append(labels[m][sel_class_idx])
            sel_units = np.concatenate(sel_units, axis=0)
            dec_feature[type].append(sel_units)

    for type in info:
        dec_feature[type] = np.concatenate(dec_feature[type], axis=1)

    return dec_feature, dec_label

# 2. Concatenate neurons across sessions

In [ ]:
## parameters
subject = 'Tir'
n_splits = 5
n_classes = 4
tps = 0
time_points = []
classifier = 'LDA'
trial_type = 'correct_trials'

# concatenate units and labels
retain_units_cat = []
update_units_cat = []
highest_units_cat = []
lowest_units_cat = []

retain_labels_cat = []
update_labels_cat = []
highest_labels_cat = []
lowest_labels_cat = []

retain_label_num = []
update_label_num = []
highest_label_num = []
lowest_label_num = []

area_all_units = {}
areas = ['PFC']
for area in areas:
    area_all_units[area] = []

min_repeated_trials = None
fixed_min = 15

path = f'/Users/huidili/Desktop/wmupdate/analysis_data/{trial_type}/'
area_path = path+'area_df/'
for filename in listdir(area_path):
    if ('area' in filename) and (subject in filename):
        if subject == 'ISA':
            sub_name, session_num, _, _ = filename.split('_')
            session_name = sub_name+'_'+session_num 
        else:
            session_name = filename.split('_')[0]
        
        area_df = pd.read_pickle(area_path+filename)       
        condition_path = path+'condition_df/'
        spk_path = path+'spike_rate/150bin_50step/'
        label_path = path+'labels/'
        condition_df = pd.read_pickle(condition_path+session_name+'_condition_df.pkl')
        spike_data = np.load(spk_path+session_name+'_spike_rate.npz')
        spike_rate = spike_data['spike_rate']
        if tps == 0:
            tps = spike_rate.shape[2]
            time_points = spike_data['rate_bins'][:,0]
            
        label_data = np.load(label_path+session_name+'_labels.npy')
        retain_trial_idx = condition_df['retain']
        update_trial_idx = condition_df['update']
        highest_trial_idx = condition_df['highest']
        lowest_trial_idx = condition_df['lowest']
        retain_labels = label_data[:, retain_trial_idx]
        update_labels = label_data[:, update_trial_idx]
        highest_labels = label_data[:,highest_trial_idx]
        lowest_labels = label_data[:, lowest_trial_idx]
        min_labels = min(np.min(count_number(retain_labels[:2])), np.min(count_number(update_labels[:2])), np.min(count_number(highest_labels[:2])), np.min(count_number(lowest_labels[:2])))


        if min_labels >= fixed_min: 
            if min_repeated_trials is None or (min_labels < min_repeated_trials):
                min_repeated_trials = min_labels
            
            for area in areas:
                area_idx = area_df[area]
                area_all_units[area].append(area_idx)
            retain_units_cat.append(spike_rate[retain_trial_idx])
            update_units_cat.append(spike_rate[update_trial_idx])
            highest_units_cat.append(spike_rate[highest_trial_idx])
            lowest_units_cat.append(spike_rate[lowest_trial_idx])
            retain_labels_cat.append(retain_labels)
            update_labels_cat.append(update_labels)
            highest_labels_cat.append(highest_labels)
            lowest_labels_cat.append(lowest_labels)

# modify the min so it can be divided by n_splits
min_repeated_trials -= min_repeated_trials % n_splits

for area in areas:
    area_all_units[area] = np.concatenate(area_all_units[area])

In [ ]:
print(f'number of sessions: {len(retain_units_cat)}')
print(f'number of trials per condition: {min_repeated_trials}')

# 3. Information decoding

In [ ]:
## spike decoding for retain and update conditions respectively
chunk_size = 10
prev_iteration = 0
end_iteration = 1000

information = ['t1', 't2']
areas = ['PFC']
conditions = ['retain', 'update']

acc_dict = {}
for area in areas: 
    for condition in conditions:
        for type in information:
            name = f'{area}_{condition}_{type}'
            acc_dict[name] = np.empty((chunk_size, n_splits, tps))


for iteration in range(prev_iteration, end_iteration):
    
    retain_dec_feature, retain_dec_label = create_pseudo_popu(retain_labels_cat, retain_units_cat, min_repeated_trials)
    update_dec_feature, update_dec_label = create_pseudo_popu(update_labels_cat, update_units_cat, min_repeated_trials)
    
    for area in areas:
        for condition in conditions:
            for type in information:
                name = f'{area}_{condition}_{type}'
                area_idx = area_all_units[area]
                if condition == 'retain': 
                    feature = retain_dec_feature[type][:,area_idx]
                    label = retain_dec_label[type]
                elif condition == 'update':
                    feature = update_dec_feature[type][:,area_idx]
                    label = update_dec_label[type]  
                
                kf = StratifiedKFold(n_splits=n_splits, shuffle=True)
                decoder = LinearDiscriminantAnalysis(priors=(1/n_classes)*np.ones((n_classes,)))
                for i_fold, (train_idx, test_idx) in enumerate(kf.split(feature, label)):
                    for t in range(feature.shape[2]):
                        feature_t = feature[:,:,t]
                        X_train, X_test = feature_t[train_idx], feature_t[test_idx]
                        y_train, y_test = label[train_idx], label[test_idx]
                        # normalization for each neurons across all training trials
                        train_mean = np.mean(X_train, axis=0)
                        train_sd = np.std(X_train, axis=0)  
                        zero_sd = (train_sd == 0)
                        train_sd[zero_sd] = math.inf
                        X_train = (X_train - train_mean) / train_sd
                        X_test = (X_test - train_mean) / train_sd
                        
                        decoder.fit(X_train, y_train)
                        
                        acc_dict[name][int(iteration%chunk_size),i_fold,t] = decoder.score(X_test, y_test)

    if (iteration+1)%chunk_size == 0:
        for area in areas:
            for condition in conditions:
                for type in information:
                    name = f'{area}_{condition}_{type}'
                    path = f'/Users/huidili/Desktop/wmupdate/analysis_result/location_decoding/1000resamples/{name}/{subject}/'
                    np.save(path+f'iteration{iteration+1}.npy', acc_dict[name])
                    acc_dict[name] = np.empty((chunk_size, n_splits, tps))
        print(f'iteration {iteration+1}')